In [14]:
import os, pickle, numpy as np, scipy.sparse as sp, torch
from torch.nn.init import xavier_normal_
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random
import itertools
import xgi

#### E is the positive edges that is originally in the hypergraph
#### F is the negative edges that is generated by both papers using their method, so everything is comparable. 
#### D is the saved data, we then extract edges from it 


### Three datasets are included in both papers so we also consider the same datasets. 

## The only thing we need to notice is that in these cases, the edges are directed, but we only consider the undirected case 




data = "iAF1260b"
#data = "iJO1366"
#data = "uspto"

#Dir = r"C:\Users\hexie\OneDrive\Desktop\Projects\hypergraph\LHP\\"
#Dir = r"C:\Users\hexie\OneDrive\Desktop\Projects\hypergraph\nhp_cikm20\nhp\data\\"
Dir = r"data\\"

with open(os.path.join(Dir, data, data + '_hypergraph.pkl'), 'rb') as h: H = pickle.load(h)


    
List_of_edge = []
List_of_nonedge = []



E, c, n = {}, 0, 0
for k in H['E']:
    t, h = H['E'][k]
    if len(t) > 0 and len(h) > 0: 
        E[c] = H['E'][k]
        for v in t.union(h): 
            if v > n: n = v
        c += 1
n = n + 1
print("Number of vertices is", n, "\nNumber of hyperlinks is", c)

cfile = os.path.join(Dir, data, data + '_candidates.pkl')
if os.path.isfile(cfile):
    with open(os.path.join(cfile), 'rb') as h: G = pickle.load(h)
def exists(tc, hc):
    for k in H['E']:
        t, h = H['E'][k]
        if t == tc and h == hc: return True
    return False


th = set()
for i in E: 
    e = E[i]
    List_of_edge.append(e)
    th.union(e[0])
    th.union(e[1])
m = len(E)

if os.path.isfile(cfile):
    C, c = {}, 0
    for k in tqdm(G['E']):
        t, h = G['E'][k]
        if len(t) > 0 and len(h) > 0 and not exists(t, h): 
            C[c] = G['E'][k]
            for v in t.union(h): 
                if v >= n: print("Something is wrong")
            c += 1
    print("Number of candidate hyperlinks is", len(C))

D = {'n': n, 'E': E, 'C': C}
F = {}
for k in tqdm(E): 
    T, H = E[k]
    
    T, H = list(T), list(H)
    sT, sH = int(len(T)/2), int(len(H)/2)
    
    rT, rH = len(T) - sT, len(H) - sH
    V = range(n)
    
    TuH = set(T).union(H)
    vT, vH = list(set(V) - TuH), list(set(V) - TuH)
    
    flag = True 
    while flag:
        nT, nH = random.sample(T, sT), random.sample(H, sH)
        oT, oH = random.sample(vT, rT), random.sample(vH, rH)
        if tuple(oT) not in th and tuple(oH) not in th and oT != oH: 
            nT, nH = set(nT+oT), set(nH+oH)
            I = set(nT).intersection(set(nH))
            if len(nT) != len(T) or len(nH) != len(H) or len(I) > 0: flag = True
            else: F[k], flag = [nT, nH], False     

if 'C' in D:
    assigned = []
    N = F.copy()
    count = 0
    for k in tqdm(D['C']):
        tc, hc = D['C'][k]
        for l in F:
            t, h = F[l]
            if len(t) == len(tc) and len(h) == len(hc) and l not in assigned:
                count += 1
                assigned.append(l)
                N[l] = D['C'][k]
                break
    F = N.copy()
    print(count, len(E), len(N))
                
                



for j in F: 
    e = F[j]
    List_of_nonedge.append(e)


# dump all the hyper-edges
#with open(data + '.pkl', 'wb') as h: pickle.dump(E, h, protocol=pickle.HIGHEST_PROTOCOL)

Number of vertices is 1668 
Number of hyperlinks is 2084


100%|████████████████████████████████████████████████████████████████████████████| 1140/1140 [00:00<00:00, 4912.31it/s]


Number of candidate hyperlinks is 1140


100%|████████████████████████████████████████████████████████████████████████████| 1140/1140 [00:00<00:00, 1616.84it/s]

1140 2084 2084


In [15]:
pos_edges = [x[0].union(x[1]) for x in List_of_edge]
neg_edges = [x[0].union(x[1]) for x in List_of_nonedge]

In [16]:
with open(data + '_pos.pkl', 'wb') as h: pickle.dump(pos_edges, h, protocol=pickle.HIGHEST_PROTOCOL)
with open(data + '_neg.pkl', 'wb') as h: pickle.dump(pos_edges, h, protocol=pickle.HIGHEST_PROTOCOL)